# Temperature, top-k and top-p: a sampling playground

*A companion to the blog post “Temperature, top-k and top-p: how models pick the next word” (Part 7 of the Tokenization & decoding series).*

A toy next-word distribution for “The capital of Japan is…”. Change the settings and watch the draws change.

In [ ]:
import numpy as np

tokens = ['Tokyo', 'Kyoto', 'Osaka', 'the', 'a', 'located', 'very', 'banana']
logits = np.array([4.0, 2.2, 1.8, 1.0, 0.2, -0.5, -1.0, -2.0])

def softmax(z):
    z = np.asarray(z, dtype=float); z = z - z.max(); e = np.exp(z); return e / e.sum()

def sample_next(logits, temperature=1.0, top_k=None, top_p=None, rng=None):
    probs = softmax(np.asarray(logits, dtype=float) / temperature)       # temperature first
    order = np.argsort(-probs)
    keep = np.ones(len(probs), dtype=bool)
    if top_k is not None:
        keep[order[top_k:]] = False
    if top_p is not None:
        cutoff = int(np.searchsorted(np.cumsum(probs[order]), top_p) + 1)  # smallest set reaching top_p
        keep[order[cutoff:]] = False
    probs = np.where(keep, probs, 0.0); probs /= probs.sum()             # renormalise what is left
    return int(rng.choice(len(probs), p=probs))

## 1. Temperature reshapes the probabilities

In [ ]:
for T in (0.5, 1.0, 2.0):
    p = softmax(logits / T)
    print(f'temperature {T}:', ' '.join(f'{t}:{v:.2f}' for t, v in zip(tokens, p)))

## 2. Draw 2,000 samples with each setting

In [ ]:
import collections
rng = np.random.default_rng(0)
def draws(**kw):
    return dict(collections.Counter(tokens[sample_next(logits, rng=rng, **kw)] for _ in range(2000)))

print('top_p=0.9, T=1  :', draws(temperature=1.0, top_p=0.9))
print('temperature 0.5  :', draws(temperature=0.5))
print('temperature 2.0  :', draws(temperature=2.0))

Low temperature almost always picks “Tokyo”; high temperature lets rare tokens such as “banana” through. Top-p removed the tail altogether.

## 3. Top-k versus top-p on a confident and an uncertain step

In [ ]:
flat = softmax([1.6, 1.5, 1.45, 1.4, 1.3, 1.2, 1.1, 1.0])      # a model that is unsure
for name, p in (('confident', softmax(logits)), ('uncertain', flat)):
    order = np.argsort(-p); cum = np.cumsum(p[order])
    n_p = int(np.searchsorted(cum, 0.9) + 1)
    print(f'{name:>9}: top-k=3 keeps {p[order[:3]].sum():.0%} of the probability; top-p=0.9 keeps {n_p} tokens')

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
for a, (name, p) in zip(ax, (('confident', softmax(logits)), ('uncertain', flat))):
    a.bar(range(8), p, color='#f5821f'); a.set_title(name); a.set_xticks(range(8)); a.set_xticklabels(range(1, 9))
plt.tight_layout(); plt.show()

**Try it:** pass `top_k=2` or `top_p=0.5` to `draws(...)`, or change a logit, and see how the draws move.

### Want the full guided version?

This notebook is a small taster from **Chapter 2 of the course *Generative AI and LLMs with Python*** (course-2-generative-ai), one of 4 world-class courses on [aibits.ecubeanalytics.com](https://aibits.ecubeanalytics.com/?utm_source=colab&utm_medium=notebook&utm_campaign=genai_ch02).

**The full guided lab, quizzes and projects are in the course.**

- [Open Chapter 2 in the course (free sign-up where the chapter is in the free first module)](https://aibits.ecubeanalytics.com/main/document/genai_gate.php?c=course-2-generative-ai&f=chapters/course-2-generative-ai/02-tokenization-lm-objective-decoding.html&utm_source=colab&utm_medium=notebook&utm_campaign=genai_ch02)
- [See the course demo](https://aibits.ecubeanalytics.com/main/document/genai_public/GENAILLMPYTHON.html?utm_source=colab&utm_medium=notebook&utm_campaign=genai_ch02)
